# Module 4: Custom Crossover, Mutation, and Selection Operators

## Learning Objectives

By completing this notebook, you will:
1. Understand how genetic operators affect search behavior
2. Implement custom crossover operators (uniform, half-uniform, feature-group)
3. Design specialized mutation operators with adaptive rates
4. Create custom selection strategies for feature selection
5. Compare operator performance empirically
6. Design operators that incorporate domain knowledge

## Prerequisites

- Module 1 completed (GA fundamentals)
- Module 4.1 completed (DEAP basics)
- Understanding of binary chromosomes
- DEAP library installed

## Estimated Time: 90 minutes

---

## 1. Why Custom Operators?

### Key Concept: Default operators may not be optimal for feature selection.

**Challenges with Standard Operators:**
1. **Feature correlations**: Related features should be selected together
2. **Sparsity**: Feature selection needs sparse solutions
3. **Convergence**: Premature convergence to suboptimal solutions
4. **Domain knowledge**: Standard operators ignore problem structure

**Custom Operators Can:**
- Preserve feature groups (e.g., one-hot encoded features)
- Encourage sparsity through specialized mutation
- Incorporate feature importance information
- Balance exploration and exploitation better

**Operator Types:**
- **Crossover**: Combine two parents to create offspring
- **Mutation**: Randomly modify individual
- **Selection**: Choose individuals for reproduction

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random
import copy

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

### 1.1 Load and Prepare Dataset

In [ ]:
# Purpose: Prepare dataset for testing custom operators
# Key Concept: Use dataset with known feature correlations

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

N_FEATURES = X_train_scaled.shape[1]

print(f"Dataset: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features: {N_FEATURES}")
print(f"\nFeature names:")
print(X.columns.tolist())

### 1.2 Setup DEAP Framework

In [ ]:
# Purpose: Initialize DEAP types
# Key Concept: Reusable across different operator experiments

# Clean up existing types
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create types
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# Define fitness function
def evaluate_features(individual):
    """Standard fitness function for comparison."""
    if sum(individual) == 0:
        return (-999.0,)
    
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_train_scaled.iloc[:, feature_mask]
    
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_selected, y_train, cv=5, scoring='accuracy')
    
    accuracy = scores.mean()
    parsimony = 0.01 * (sum(individual) / len(individual))
    
    return (accuracy - parsimony,)

print("DEAP framework initialized.")

## 2. Custom Crossover Operators

### Key Concept: Crossover combines parent solutions to create offspring.

### 2.1 Uniform Crossover

Each gene is independently inherited from either parent with 50% probability.

In [ ]:
# Purpose: Implement uniform crossover
# Key Concept: High disruption, good for exploration

def uniform_crossover(ind1, ind2, indpb=0.5):
    """
    Uniform crossover operator.
    
    Each gene is independently swapped with probability indpb.
    
    Parameters
    ----------
    ind1, ind2 : Individual
        Parents to crossover
    indpb : float
        Probability of swapping each gene
    
    Returns
    -------
    ind1, ind2 : Individual
        Modified offspring (DEAP modifies in-place)
    """
    for i in range(len(ind1)):
        if random.random() < indpb:
            # Swap genes
            ind1[i], ind2[i] = ind2[i], ind1[i]
    
    return ind1, ind2

# Test uniform crossover
parent1 = creator.Individual([1, 1, 1, 0, 0, 0, 1, 1, 0, 0])
parent2 = creator.Individual([0, 0, 1, 1, 1, 0, 0, 1, 1, 1])

print("Uniform Crossover Example:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

child1, child2 = uniform_crossover(copy.deepcopy(parent1), copy.deepcopy(parent2))

print(f"Child 1:  {child1}")
print(f"Child 2:  {child2}")

### 2.2 Half-Uniform Crossover (HUX)

Swaps exactly half of the differing genes. Promotes diversity while maintaining structure.

In [ ]:
# Purpose: Implement HUX crossover
# Key Concept: Controlled disruption, maintains Hamming distance

def hux_crossover(ind1, ind2):
    """
    Half-Uniform Crossover (HUX).
    
    Swaps exactly half of the genes that differ between parents.
    Maintains maximum diversity in offspring.
    
    Parameters
    ----------
    ind1, ind2 : Individual
        Parents to crossover
    
    Returns
    -------
    ind1, ind2 : Individual
        Modified offspring
    """
    # Find differing positions
    diff_positions = [i for i in range(len(ind1)) if ind1[i] != ind2[i]]
    
    # Swap half of them
    n_swap = len(diff_positions) // 2
    
    if n_swap > 0:
        swap_positions = random.sample(diff_positions, n_swap)
        
        for pos in swap_positions:
            ind1[pos], ind2[pos] = ind2[pos], ind1[pos]
    
    return ind1, ind2

# Test HUX
parent1 = creator.Individual([1, 1, 1, 0, 0, 0, 1, 1, 0, 0])
parent2 = creator.Individual([0, 0, 1, 1, 1, 0, 0, 1, 1, 1])

print("HUX Crossover Example:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Calculate differences
differences = sum([1 for i in range(len(parent1)) if parent1[i] != parent2[i]])
print(f"Differing genes: {differences}")

child1, child2 = hux_crossover(copy.deepcopy(parent1), copy.deepcopy(parent2))

print(f"Child 1:  {child1}")
print(f"Child 2:  {child2}")

# Verify half were swapped
new_diff = sum([1 for i in range(len(child1)) if child1[i] != child2[i]])
print(f"Differing genes after HUX: {new_diff} (should be same: {differences})")

### 2.3 Feature-Group Crossover

Respects feature groups (e.g., one-hot encoded features, polynomial features).

In [ ]:
# Purpose: Implement feature-group aware crossover
# Key Concept: Preserve domain structure in crossover

def feature_group_crossover(ind1, ind2, feature_groups):
    """
    Crossover that respects feature groupings.
    
    Entire feature groups are inherited from one parent or the other,
    rather than mixing genes within groups.
    
    Parameters
    ----------
    ind1, ind2 : Individual
        Parents to crossover
    feature_groups : list of lists
        Each sublist contains indices of features in a group
    
    Returns
    -------
    ind1, ind2 : Individual
        Modified offspring
    """
    for group in feature_groups:
        # Randomly decide whether to swap this group
        if random.random() < 0.5:
            # Swap entire group
            for idx in group:
                ind1[idx], ind2[idx] = ind2[idx], ind1[idx]
    
    return ind1, ind2

# Test feature-group crossover
# For breast cancer data, features are grouped by type (mean, SE, worst)
# E.g., features 0-9 are means, 10-19 are SE, 20-29 are worst
feature_groups = [
    list(range(0, 10)),   # Mean features
    list(range(10, 20)),  # SE features
    list(range(20, 30))   # Worst features
]

parent1 = creator.Individual([1] * 10 + [0] * 10 + [1] * 10)
parent2 = creator.Individual([0] * 10 + [1] * 10 + [0] * 10)

print("Feature-Group Crossover Example:")
print(f"Parent 1: {parent1[:10]} | {parent1[10:20]} | {parent1[20:30]}")
print(f"Parent 2: {parent2[:10]} | {parent2[10:20]} | {parent2[20:30]}")

child1, child2 = feature_group_crossover(
    copy.deepcopy(parent1), 
    copy.deepcopy(parent2),
    feature_groups
)

print(f"Child 1:  {child1[:10]} | {child1[10:20]} | {child1[20:30]}")
print(f"Child 2:  {child2[:10]} | {child2[10:20]} | {child2[20:30]}")
print("\nNote: Entire groups are swapped together, not individual features.")

## 3. Custom Mutation Operators

### Key Concept: Mutation introduces new genetic material and prevents stagnation.

### 3.1 Adaptive Mutation

Mutation rate adapts based on population diversity or generation number.

In [ ]:
# Purpose: Implement adaptive mutation rate
# Key Concept: High mutation early (exploration), low mutation late (exploitation)

class AdaptiveMutation:
    """
    Adaptive mutation with decreasing rate.
    """
    
    def __init__(self, indpb_start=0.1, indpb_end=0.01, max_generations=50):
        """
        Parameters
        ----------
        indpb_start : float
            Initial mutation probability per gene
        indpb_end : float
            Final mutation probability per gene
        max_generations : int
            Total generations for decay schedule
        """
        self.indpb_start = indpb_start
        self.indpb_end = indpb_end
        self.max_generations = max_generations
        self.current_generation = 0
    
    def __call__(self, individual):
        """
        Mutate individual with adaptive rate.
        
        Parameters
        ----------
        individual : Individual
            Individual to mutate
        
        Returns
        -------
        individual : Individual
            Mutated individual (tuple required by DEAP)
        """
        # Calculate current mutation rate (linear decay)
        progress = min(self.current_generation / self.max_generations, 1.0)
        indpb = self.indpb_start + (self.indpb_end - self.indpb_start) * progress
        
        # Mutate
        for i in range(len(individual)):
            if random.random() < indpb:
                individual[i] = 1 - individual[i]  # Flip bit
        
        return (individual,)
    
    def update_generation(self, gen):
        """Update current generation."""
        self.current_generation = gen

# Test adaptive mutation
adaptive_mut = AdaptiveMutation(indpb_start=0.2, indpb_end=0.01, max_generations=50)

print("Adaptive Mutation Rate Over Generations:")
print("="*50)

for gen in [0, 10, 25, 40, 50]:
    adaptive_mut.update_generation(gen)
    progress = min(gen / 50, 1.0)
    rate = 0.2 + (0.01 - 0.2) * progress
    print(f"Generation {gen:2d}: mutation rate = {rate:.4f}")

### 3.2 Sparsity-Promoting Mutation

Bias mutation toward turning genes OFF to encourage sparse solutions.

In [ ]:
# Purpose: Implement sparsity-promoting mutation
# Key Concept: Favor fewer features through biased mutation

def sparsity_mutation(individual, indpb=0.05, sparsity_bias=0.7):
    """
    Mutation biased toward turning genes OFF.
    
    When a gene is selected for mutation, it's turned OFF with
    probability sparsity_bias, ON with probability (1 - sparsity_bias).
    
    Parameters
    ----------
    individual : Individual
        Individual to mutate
    indpb : float
        Probability of mutating each gene
    sparsity_bias : float
        Probability of turning gene OFF (vs ON) when mutating
    
    Returns
    -------
    individual : Individual
        Mutated individual
    """
    for i in range(len(individual)):
        if random.random() < indpb:
            # Mutate this gene
            if random.random() < sparsity_bias:
                individual[i] = 0  # Turn OFF
            else:
                individual[i] = 1  # Turn ON
    
    return (individual,)

# Test sparsity mutation
test_ind = creator.Individual([1] * 20 + [0] * 10)

print("Sparsity Mutation Test:")
print(f"Original: {sum(test_ind)} features ON")

# Run multiple mutations
for trial in range(5):
    mutated = copy.deepcopy(test_ind)
    sparsity_mutation(mutated, indpb=0.1, sparsity_bias=0.7)
    print(f"Trial {trial+1}:  {sum(mutated)} features ON (change: {sum(mutated) - sum(test_ind):+d})")

print("\nNote: Sparsity bias tends to reduce number of features.")

### 3.3 Guided Mutation

Use feature importance to guide mutation probability.

In [ ]:
# Purpose: Implement importance-guided mutation
# Key Concept: Protect important features from mutation

def compute_feature_importance(X_data, y_data):
    """
    Compute feature importance using random forest.
    
    Returns
    -------
    importances : np.ndarray
        Normalized feature importances (sum to 1)
    """
    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X_data, y_data)
    
    importances = model.feature_importances_
    importances = importances / importances.sum()  # Normalize
    
    return importances

def guided_mutation(individual, feature_importance, base_indpb=0.05, importance_weight=2.0):
    """
    Mutation with probability inversely related to feature importance.
    
    Important features are mutated less frequently.
    
    Parameters
    ----------
    individual : Individual
        Individual to mutate
    feature_importance : array-like
        Importance scores for each feature
    base_indpb : float
        Base mutation probability
    importance_weight : float
        Weight for importance adjustment (higher = stronger protection)
    
    Returns
    -------
    individual : Individual
        Mutated individual
    """
    for i in range(len(individual)):
        # Calculate mutation probability for this feature
        # Less important features have higher mutation probability
        indpb_i = base_indpb * (1.0 + importance_weight * (1.0 - feature_importance[i]))
        indpb_i = min(indpb_i, 1.0)  # Cap at 1.0
        
        if random.random() < indpb_i:
            individual[i] = 1 - individual[i]
    
    return (individual,)

# Compute feature importance
feature_importance = compute_feature_importance(X_train_scaled, y_train)

print("Feature Importance (top 10):")
top_features = np.argsort(feature_importance)[::-1][:10]
for i in top_features:
    print(f"  {X.columns[i][:30]:30s}: {feature_importance[i]:.4f}")

print("\nGuided mutation protects important features from random changes.")

## 4. Custom Selection Operators

### Key Concept: Selection determines which individuals reproduce.

### 4.1 Diversity-Preserving Selection

Balance fitness and diversity to prevent premature convergence.

In [ ]:
# Purpose: Implement diversity-aware selection
# Key Concept: Penalize similar individuals to maintain population diversity

def diversity_selection(individuals, k, fitness_weight=0.7):
    """
    Select individuals based on fitness and diversity.
    
    Selection score = fitness_weight * fitness + (1 - fitness_weight) * diversity
    
    Parameters
    ----------
    individuals : list of Individual
        Population to select from
    k : int
        Number of individuals to select
    fitness_weight : float
        Weight for fitness (vs diversity)
    
    Returns
    -------
    selected : list of Individual
        Selected individuals
    """
    # Calculate fitness scores
    fitnesses = np.array([ind.fitness.values[0] for ind in individuals])
    
    # Normalize fitness to [0, 1]
    if fitnesses.max() != fitnesses.min():
        fitnesses_norm = (fitnesses - fitnesses.min()) / (fitnesses.max() - fitnesses.min())
    else:
        fitnesses_norm = np.ones_like(fitnesses)
    
    # Calculate diversity scores (average Hamming distance to others)
    diversity_scores = np.zeros(len(individuals))
    
    for i, ind_i in enumerate(individuals):
        hamming_distances = []
        for j, ind_j in enumerate(individuals):
            if i != j:
                # Hamming distance
                dist = sum([1 for k in range(len(ind_i)) if ind_i[k] != ind_j[k]])
                hamming_distances.append(dist)
        
        diversity_scores[i] = np.mean(hamming_distances) if hamming_distances else 0
    
    # Normalize diversity to [0, 1]
    if diversity_scores.max() != diversity_scores.min():
        diversity_norm = (diversity_scores - diversity_scores.min()) / \
                        (diversity_scores.max() - diversity_scores.min())
    else:
        diversity_norm = np.ones_like(diversity_scores)
    
    # Combined score
    scores = fitness_weight * fitnesses_norm + (1 - fitness_weight) * diversity_norm
    
    # Select top k
    selected_indices = np.argsort(scores)[::-1][:k]
    selected = [individuals[i] for i in selected_indices]
    
    return selected

print("Diversity-preserving selection implemented.")
print("This selection balances fitness with population diversity.")

### 4.2 Lexicographic Selection

Primary criterion: fitness. Secondary criterion: number of features (parsimony).

In [ ]:
# Purpose: Implement lexicographic selection
# Key Concept: Among equally fit individuals, prefer simpler ones

def lexicographic_selection(individuals, k, fitness_tolerance=0.01):
    """
    Lexicographic selection: fitness first, then parsimony.
    
    Select based on fitness. Among individuals within fitness_tolerance,
    prefer those with fewer features.
    
    Parameters
    ----------
    individuals : list of Individual
        Population to select from
    k : int
        Number of individuals to select
    fitness_tolerance : float
        Fitness difference threshold for applying parsimony
    
    Returns
    -------
    selected : list of Individual
        Selected individuals
    """
    # Sort by fitness (descending)
    sorted_inds = sorted(individuals, key=lambda ind: ind.fitness.values[0], reverse=True)
    
    selected = []
    
    while len(selected) < k and sorted_inds:
        # Get best remaining individual
        best = sorted_inds[0]
        best_fitness = best.fitness.values[0]
        
        # Find all individuals within tolerance
        candidates = [ind for ind in sorted_inds 
                     if abs(ind.fitness.values[0] - best_fitness) <= fitness_tolerance]
        
        # Among candidates, select one with fewest features
        chosen = min(candidates, key=lambda ind: sum(ind))
        selected.append(chosen)
        
        # Remove chosen from sorted list
        sorted_inds.remove(chosen)
    
    return selected

print("Lexicographic selection implemented.")
print("This promotes sparse solutions when fitness is similar.")

## 5. Operator Comparison Experiment

### Key Concept: Empirically compare different operators on same problem.

In [ ]:
# Purpose: Run controlled experiments comparing operators
# Key Concept: Hold other parameters constant, vary one operator

def run_ga_experiment(crossover_func, mutation_func, selection_func, 
                      n_generations=30, pop_size=40, exp_name="Experiment"):
    """
    Run GA with specified operators.
    
    Returns
    -------
    results : dict
        Best fitness, convergence generation, best solution
    """
    # Create toolbox
    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register(
        "individual",
        tools.initRepeat,
        creator.Individual,
        toolbox.attr_bool,
        n=N_FEATURES
    )
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_features)
    toolbox.register("mate", crossover_func)
    toolbox.register("mutate", mutation_func)
    toolbox.register("select", selection_func)
    
    # Statistics
    stats = tools.Statistics(key=lambda ind: ind.fitness.values)
    stats.register("max", np.max)
    stats.register("avg", np.mean)
    
    hall_of_fame = tools.HallOfFame(1)
    
    # Run GA
    population = toolbox.population(n=pop_size)
    
    population, logbook = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.7,
        mutpb=0.2,
        ngen=n_generations,
        stats=stats,
        halloffame=hall_of_fame,
        verbose=False
    )
    
    # Extract results
    best_ind = hall_of_fame[0]
    best_fitness = best_ind.fitness.values[0]
    n_features = sum(best_ind)
    
    # Find convergence generation (when improvement < 0.001)
    max_fits = logbook.select("max")
    convergence_gen = n_generations
    for i in range(1, len(max_fits)):
        if max_fits[i] - max_fits[i-1] < 0.001:
            convergence_gen = i
            break
    
    return {
        'name': exp_name,
        'best_fitness': best_fitness,
        'n_features': n_features,
        'convergence_gen': convergence_gen,
        'logbook': logbook
    }

print("Experiment framework ready.")

### 5.1 Run Crossover Comparison

In [ ]:
# Purpose: Compare different crossover operators
# Key Concept: Keep mutation and selection constant

print("Running crossover comparison experiments...\n")

experiments = []

# Experiment 1: Two-point crossover (baseline)
print("1. Two-Point Crossover")
result = run_ga_experiment(
    crossover_func=tools.cxTwoPoint,
    mutation_func=lambda ind: tools.mutFlipBit(ind, indpb=0.05),
    selection_func=lambda inds, k: tools.selTournament(inds, k, tournsize=3),
    exp_name="Two-Point"
)
experiments.append(result)
print(f"   Best fitness: {result['best_fitness']:.4f}, Features: {result['n_features']}")

# Experiment 2: Uniform crossover
print("2. Uniform Crossover")
result = run_ga_experiment(
    crossover_func=uniform_crossover,
    mutation_func=lambda ind: tools.mutFlipBit(ind, indpb=0.05),
    selection_func=lambda inds, k: tools.selTournament(inds, k, tournsize=3),
    exp_name="Uniform"
)
experiments.append(result)
print(f"   Best fitness: {result['best_fitness']:.4f}, Features: {result['n_features']}")

# Experiment 3: HUX crossover
print("3. HUX Crossover")
result = run_ga_experiment(
    crossover_func=hux_crossover,
    mutation_func=lambda ind: tools.mutFlipBit(ind, indpb=0.05),
    selection_func=lambda inds, k: tools.selTournament(inds, k, tournsize=3),
    exp_name="HUX"
)
experiments.append(result)
print(f"   Best fitness: {result['best_fitness']:.4f}, Features: {result['n_features']}")

print("\nExperiments completed!")

### 5.2 Visualize Comparison

In [ ]:
# Purpose: Visualize operator comparison results
# Key Concept: Compare convergence speed and final performance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Convergence curves
for exp in experiments:
    logbook = exp['logbook']
    gen = logbook.select("gen")
    fit_max = logbook.select("max")
    axes[0].plot(gen, fit_max, linewidth=2, label=exp['name'], marker='o', markersize=4)

axes[0].set_xlabel('Generation', fontsize=12)
axes[0].set_ylabel('Best Fitness', fontsize=12)
axes[0].set_title('Convergence Comparison: Crossover Operators', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Plot 2: Final results comparison
names = [exp['name'] for exp in experiments]
fitness_vals = [exp['best_fitness'] for exp in experiments]
n_features_vals = [exp['n_features'] for exp in experiments]

x = np.arange(len(names))
width = 0.35

ax2_twin = axes[1].twinx()
bars1 = axes[1].bar(x - width/2, fitness_vals, width, label='Best Fitness', color='steelblue')
bars2 = ax2_twin.bar(x + width/2, n_features_vals, width, label='Num Features', color='coral')

axes[1].set_xlabel('Crossover Operator', fontsize=12)
axes[1].set_ylabel('Best Fitness', fontsize=12, color='steelblue')
ax2_twin.set_ylabel('Number of Features', fontsize=12, color='coral')
axes[1].set_title('Final Performance Comparison', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names)
axes[1].tick_params(axis='y', labelcolor='steelblue')
ax2_twin.tick_params(axis='y', labelcolor='coral')
axes[1].grid(alpha=0.3, axis='y')

# Add legends
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

print("\nComparison Summary:")
for exp in experiments:
    print(f"  {exp['name']:12s}: Fitness={exp['best_fitness']:.4f}, "
          f"Features={exp['n_features']:2d}, Converged at gen {exp['convergence_gen']}")

## 6. Exercises

### Exercise 6.1: Implement Three-Point Crossover

**Task**: Implement three-point crossover (three cut points, alternating segments) and compare with two-point.

**Expected Output**: Analysis of disruption level and performance.

<details>
<summary>Hint 1</summary>
Randomly select 3 cut points, sort them, then alternate copying from parent1 and parent2.
</details>

<details>
<summary>Hint 2</summary>
Use `random.sample(range(len(ind)), 3)` to get cut points.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def three_point_crossover(ind1, ind2):
    """
    Three-point crossover operator.
    """
    # TODO: Implement
    pass

### Exercise 6.2: Diversity-Driven Adaptive Mutation

**Task**: Implement mutation that increases rate when population diversity drops below threshold.

**Expected Output**: Mutation rate responds to population state.

<details>
<summary>Hint</summary>
Measure diversity as average Hamming distance in population. Increase mutation when diversity < threshold.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class DiversityAdaptiveMutation:
    """
    Mutation rate adapts based on population diversity.
    """
    
    def __init__(self, base_indpb=0.05, min_diversity=10.0, boost_factor=2.0):
        # TODO: Implement
        pass
    
    def __call__(self, individual, population):
        # TODO: Implement
        pass

### Exercise 6.3: Rank-Based Selection with Pressure Control

**Task**: Implement rank-based selection where selection pressure can be controlled via parameter.

**Expected Output**: Adjustable selection pressure affects convergence speed.

<details>
<summary>Hint</summary>
Assign probabilities based on rank: `prob[i] = (2 - pressure) / N + 2 * i * (pressure - 1) / (N * (N - 1))`
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def rank_selection(individuals, k, pressure=1.5):
    """
    Rank-based selection with controllable pressure.
    
    Parameters
    ----------
    pressure : float
        Selection pressure (1.0 = uniform, 2.0 = maximum)
    """
    # TODO: Implement
    pass

## 7. Summary

### Key Takeaways

1. **Operator Impact**: Choice of operators significantly affects GA performance
2. **Custom Design**: Problem-specific operators often outperform generic ones
3. **Crossover Types**: Uniform (high disruption), two-point (medium), HUX (controlled)
4. **Mutation Strategies**: Adaptive, sparsity-promoting, guided by importance
5. **Selection Balance**: Trade-off between selection pressure and diversity
6. **Domain Knowledge**: Incorporate problem structure (feature groups, importance)

### Operator Design Principles

**Crossover:**
- Exploit problem structure (feature groups)
- Balance disruption level
- Preserve good building blocks

**Mutation:**
- Adapt to search progress
- Maintain diversity
- Use domain knowledge
- Favor sparse solutions when appropriate

**Selection:**
- Avoid premature convergence
- Maintain population diversity
- Apply secondary criteria (parsimony, novelty)

### Operator Comparison Guidelines

1. **Controlled experiments**: Change one operator at a time
2. **Multiple runs**: Account for stochasticity (run 10+ times)
3. **Multiple metrics**: Fitness, convergence speed, diversity, sparsity
4. **Statistical testing**: Use t-tests or Wilcoxon tests for significance
5. **Problem variety**: Test on multiple datasets

### When to Use Custom Operators

- **Feature groups exist**: Use group-aware crossover
- **Sparsity desired**: Use sparsity-promoting mutation
- **Prior knowledge available**: Use guided mutation
- **Premature convergence**: Use diversity-preserving selection
- **Slow convergence**: Increase selection pressure

### What's Next

- **Next notebook**: Complete case study with high-dimensional data
- **Advanced**: Multi-objective operators, co-evolution
- **Production**: A/B test operators in deployed systems

### Additional Resources

- **Crossover Analysis**: Spears & De Jong (1991) - "On the virtues of parameterized uniform crossover"
- **Adaptive Operators**: Eiben et al. (1999) - "Parameter control in evolutionary algorithms"
- **Selection Methods**: Goldberg & Deb (1991) - "A comparative analysis of selection schemes"